<a href="https://colab.research.google.com/github/SharikaTasnim23/ECE_20/blob/main/Flood_risk_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install openmeteo-requests
!pip install requests-cache retry-requests numpy pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.7/208.7 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 723.5/723.5 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 399.4/399.4 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 58.3 MB/s eta 0:00:00
  Attempting uninstall: flatbuffers
    Found existing installation: flatbuffers 25.12.19
    Uninstalling flatbuffers-25.12.19:
      Successfully uninstalled flatbuffers-25.12.19
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.2/70.2 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.6 MB/s eta 0:00:00


In [ ]:
import requests

# Test Open-Meteo Archive API
def test_open_meteo_archive():
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": 23.685,  # Example coordinates for Bangladesh
        "longitude": 90.3563,
        "start_date": "2022-01-01",
        "end_date": "2022-12-31",
        "daily": ["precipitation_sum", "temperature_2m_max"],
        "timezone": "Asia/Dhaka"
    }
    response = requests.get(url, params=params)
    if response.status_code == 200:
        print("Open-Meteo Archive API is working!")
        print(response.json())  # Display the response data
    else:
        print(f"Failed to fetch data: {response.status_code}")

# Call the test functions
test_open_meteo_archive()

Open-Meteo Archive API is working!
{'latitude': 23.655535, 'longitude': 90.379745, 'generationtime_ms': 8.920669555664062, 'utc_offset_seconds': 21600, 'timezone': 'Asia/Dhaka', 'timezone_abbreviation': 'GMT+6', 'elevation': 3.0, 'daily_units': {'time': 'iso8601', 'precipitation_sum': 'mm', 'temperature_2m_max': '°C'}, 'daily': {'time': ['2022-01-01', '2022-01-02', '2022-01-03', '2022-01-04', '2022-01-05', '2022-01-06', '2022-01-07', '2022-01-08', '2022-01-09', '2022-01-10', '2022-01-11', '2022-01-12', '2022-01-13', '2022-01-14', '2022-01-15', '2022-01-16', '2022-01-17', '2022-01-18', '2022-01-19', '2022-01-20', '2022-01-21', '2022-01-22', '2022-01-23', '2022-01-24', '2022-01-25', '2022-01-26', '2022-01-27', '2022-01-28', '2022-01-29', '2022-01-30', '2022-01-31', '2022-02-01', '2022-02-02', '2022-02-03', '2022-02-04', '2022-02-05', '2022-02-06', '2022-02-07', '2022-02-08', '2022-02-09', '2022-02-10', '2022-02-11', '2022-02-12', '2022-02-13', '2022-02-14', '2022-02-15', '2022-02-16', '2

In [ ]:
# Test Open-Meteo Forecast API
def test_open_meteo_forecast():
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": 23.685,  # Example coordinates for Bangladesh
        "longitude": 90.3563,
        "daily": ["precipitation_sum", "temperature_2m_max"],
        "timezone": "Asia/Dhaka"
    }
    response = requests.get(url, params=params)
    if response.status_code == 200:
        print("Open-Meteo Forecast API is working!")
        print(response.json())  # Display the response data
    else:
        print(f"Failed to fetch data: {response.status_code}")

test_open_meteo_forecast()

Open-Meteo Forecast API is working!
{'latitude': 23.625, 'longitude': 90.375, 'generationtime_ms': 0.03421306610107422, 'utc_offset_seconds': 21600, 'timezone': 'Asia/Dhaka', 'timezone_abbreviation': 'GMT+6', 'elevation': 3.0, 'daily_units': {'time': 'iso8601', 'precipitation_sum': 'mm', 'temperature_2m_max': '°C'}, 'daily': {'time': ['2026-04-17', '2026-04-18', '2026-04-19', '2026-04-20', '2026-04-21', '2026-04-22', '2026-04-23'], 'precipitation_sum': [5.0, 0.1, 0.6, 5.7, 2.7, 0.0, 0.0], 'temperature_2m_max': [33.4, 33.0, 33.7, 35.9, 36.2, 35.4, 34.4]}}


In [ ]:
!pip install xgboost tensorflow --quiet

In [ ]:
import os, time, json, warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import numpy as np
import pandas as pd
import requests
from datetime import datetime, timedelta
from pathlib import Path

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import roc_auc_score, f1_score, classification_report

import xgboost as xgb
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

print(f"TensorFlow : {tf.__version__}")
print(f"XGBoost    : {xgb.__version__}")
print("Imports done.")

TensorFlow : 2.19.0
XGBoost    : 3.2.0
Imports done.


In [ ]:
TRAIN_START     = "2015-01-01"
TRAIN_END       = "2022-12-31"
VAL_START       = "2023-01-01"
VAL_END         = "2023-12-31"
SEQUENCE_LENGTH = 30
GRID_ROWS       = 24
GRID_COLS       = 22
N_CHANNELS      = 7          # added slope channel vs v1
CACHE_DIR       = Path("osm_cache")
CACHE_DIR.mkdir(exist_ok=True)

# Bangladesh bounding box [south, west, north, east]
BD_BBOX = (20.5, 87.9, 26.7, 92.7)

# Upstream catchments — these cannot come from OSM Bangladesh bbox
# because they are physically outside Bangladesh.
# WHY THEY MATTER: Brahmaputra rain in Assam takes 2-5 days to arrive.
UPSTREAM_FIXED = {
    'Assam_Dhubri':       {'lat': 26.022, 'lon': 89.975},
    'Meghalaya_Shillong': {'lat': 25.578, 'lon': 91.893},
    'Nepal_Birganj':      {'lat': 27.014, 'lon': 84.877},
}

# API endpoints
OVERPASS_URL    = "https://overpass-api.de/api/interpreter"
TOPO_URL        = "https://api.opentopodata.org/v1/srtm30m"
METEO_ARCHIVE   = "https://archive-api.open-meteo.com/v1/archive"
METEO_FORECAST  = "https://api.open-meteo.com/v1/forecast"
NOMINATIM_URL   = "https://nominatim.openstreetmap.org/search"


In [ ]:
def fetch_osm_rivers(bbox=BD_BBOX, cache_file=CACHE_DIR / "osm_rivers.json"):
    """
    Queries OSM Overpass for all named rivers inside Bangladesh.
    Returns a list of {name, lat, lon} dicts.
    Uses disk cache — first call takes ~60s, subsequent calls instant.
    """
    if cache_file.exists():
        print("  Loading OSM rivers from cache ...")
        return json.loads(cache_file.read_text())

    print("  Querying OSM Overpass API for Bangladesh rivers ...")
    south, west, north, east = bbox

    # Overpass QL query:
    # Find all 'way' and 'node' elements tagged as waterway=river
    # inside the Bangladesh bounding box, then output their coords.
    query = f"""
    [out:json][timeout:60];
    (
      way["waterway"~"river|stream"]["name"]
         ({south},{west},{north},{east});
    );
    out center tags;
    """
    try:
        r = requests.post(OVERPASS_URL, data={'data': query}, timeout=90)
        r.raise_for_status()
        data = r.json()
    except Exception as e:
        print(f"  Overpass failed: {e}. Using fallback stations.")
        return _fallback_stations()

    rivers = {}
    for el in data.get('elements', []):
        name = el.get('tags', {}).get('name', 'Unknown')
        center = el.get('center', {})
        if center.get('lat') and center.get('lon'):
            if name not in rivers:
                rivers[name] = []
            rivers[name].append({'lat': center['lat'], 'lon': center['lon']})

    # Sample one point per ~0.3° along each major river
    stations = []
    for river_name, points in rivers.items():
        pts = sorted(points, key=lambda p: p['lat'])
        sampled, last_lat = [], -99
        for p in pts:
            if p['lat'] - last_lat > 0.28:
                sampled.append({'name': river_name, 'lat': p['lat'], 'lon': p['lon']})
                last_lat = p['lat']
        stations.extend(sampled[:8])   # cap per river to avoid too many API calls

    print(f"  Discovered {len(stations)} monitoring points across {len(rivers)} rivers.")
    cache_file.write_text(json.dumps(stations))
    return stations


def fetch_elevations(stations, cache_file=CACHE_DIR / "elevations.json"):
    """
    Calls OpenTopoData SRTM-30m API to get real ground elevation
    for every monitoring point.

    API: https://api.opentopodata.org/v1/srtm30m?locations=lat,lon|lat,lon
    Free: YES — no key, 100 points per request, 1 request/second

    WHY ELEVATION IS CRITICAL FOR PROBLEM 1+2:
    A place at 2m elevation near a river will flood with 50mm of
    rain. The same rainfall at 40m elevation will not flood.
    This is the physics knowledge that generalises to areas with
    no flood history.
    """
    if cache_file.exists():
        print("  Loading elevations from cache ...")
        elev_map = json.loads(cache_file.read_text())
        for st in stations:
            key = f"{st['lat']:.4f},{st['lon']:.4f}"
            st['elevation_m'] = elev_map.get(key, 10.0)
        return stations

    print("  Fetching SRTM elevations from OpenTopoData ...")
    elev_map = {}
    batch_size = 80   # API allows 100; use 80 to stay safe

    for i in range(0, len(stations), batch_size):
        batch = stations[i:i + batch_size]
        locs  = "|".join(f"{s['lat']:.4f},{s['lon']:.4f}" for s in batch)
        try:
            r = requests.get(TOPO_URL, params={'locations': locs}, timeout=30)
            r.raise_for_status()
            results = r.json().get('results', [])
            for st, res in zip(batch, results):
                key = f"{st['lat']:.4f},{st['lon']:.4f}"
                elev_map[key] = res.get('elevation', 10.0)
        except Exception as e:
            print(f"  Elevation batch failed: {e}")
            for st in batch:
                key = f"{st['lat']:.4f},{st['lon']:.4f}"
                elev_map[key] = 10.0
        time.sleep(1.1)   # respect 1 req/s rate limit

    for st in stations:
        key = f"{st['lat']:.4f},{st['lon']:.4f}"
        st['elevation_m'] = elev_map.get(key, 10.0)

    cache_file.write_text(json.dumps(elev_map))
    return stations


def compute_slope(stations):
    """
    Estimates terrain slope for each monitoring point by comparing
    its elevation to its nearest neighbours.

    WHY SLOPE MATTERS FOR PROBLEM 1+2:
    Flat land (slope ~0°) traps water. Steep land (slope >5°) drains
    it. A never-flooded area on a flat river delta with 1m elevation
    is at extreme risk even without any historical flood record.
    """
    if len(stations) < 3:
        for s in stations:
            s['slope'] = 0.5
        return stations

    lats = np.array([s['lat'] for s in stations])
    lons = np.array([s['lon'] for s in stations])
    elevs = np.array([s['elevation_m'] for s in stations])

    for i, s in enumerate(stations):
        # Find 4 nearest neighbours
        dists = np.sqrt((lats - s['lat'])**2 + (lons - s['lon'])**2)
        dists[i] = 999
        nn_idx = np.argsort(dists)[:4]
        if len(nn_idx) == 0:
            s['slope'] = 0.5
            continue
        elev_diffs = np.abs(elevs[nn_idx] - s['elevation_m'])
        dist_deg   = dists[nn_idx]
        dist_m     = dist_deg * 111_000   # approx metres per degree
        slopes     = elev_diffs / (dist_m + 1)
        s['slope'] = float(np.mean(slopes))

    return stations


def _fallback_stations():
    """
    Used if Overpass API is down. Returns the same 12 districts
    as version 1 but WITHOUT static risk_zone — just lat/lon.
    """
    return [
        {'name': 'Brahmaputra', 'lat': 25.807, 'lon': 89.636},
        {'name': 'Brahmaputra', 'lat': 25.328, 'lon': 89.528},
        {'name': 'Jamuna',      'lat': 24.937, 'lon': 89.937},
        {'name': 'Jamuna',      'lat': 24.451, 'lon': 89.701},
        {'name': 'Jamuna',      'lat': 24.251, 'lon': 89.917},
        {'name': 'Surma',       'lat': 25.069, 'lon': 91.399},
        {'name': 'Surma',       'lat': 24.899, 'lon': 91.872},
        {'name': 'Kongsha',     'lat': 24.870, 'lon': 90.727},
        {'name': 'Padma',       'lat': 23.864, 'lon': 90.000},
        {'name': 'Padma',       'lat': 24.363, 'lon': 88.601},
        {'name': 'Buriganga',   'lat': 23.724, 'lon': 90.408},
        {'name': 'Meghna',      'lat': 23.200, 'lon': 90.700},
    ]

In [ ]:
def compute_physics_features(df, elevation_m, slope, river_name="unknown",
                              seasonal_rain_mean=None):
    """
    Converts raw weather data into physics-based flood features.
    Works for ANY lat/lon — no historical flood label needed.

    Parameters:
        df               : DataFrame with 'rainfall_mm', 'soil_moisture', etc.
        elevation_m      : Ground elevation in metres from SRTM
        slope            : Terrain slope estimate (0 = flat, higher = steep)
        river_name       : Name of nearby river (used to estimate river width)
        seasonal_rain_mean: Long-run monthly mean rainfall (for anomaly calc)
    """
    df = df.copy().sort_index()

    # ── PHYSICS FEATURE 1: Elevation (direct input) ──
    # A point at 1m is far more vulnerable than one at 30m.
    # We normalise: low_elev_score = 1 for 0m, 0 for 100m+
    df['elevation_m']      = elevation_m
    df['low_elev_score']   = 1.0 / (1.0 + elevation_m / 15.0)

    # ── PHYSICS FEATURE 2: Terrain slope ──
    df['slope']            = slope
    df['flat_land_score']  = 1.0 / (1.0 + slope * 5000)

    # ── PHYSICS FEATURE 3: Flood Susceptibility Index ──
    # Combines elevation vulnerability + flat-land trapping potential.
    # A river-proximate low flat point scores close to 1.
    # A highland steep point scores close to 0.
    # Note: river_proximity is baked in at prediction time via upstream features.
    df['flood_susceptibility_idx'] = df['low_elev_score'] * df['flat_land_score']

    # ── PHYSICS FEATURE 4: Soil Saturation Index ──
    if 'soil_moisture' not in df.columns:
        df['soil_moisture'] = 0.5
    df['soil_moisture'] = df['soil_moisture'].fillna(method='ffill').fillna(0.5)
    rain_14d = df['rainfall_mm'].fillna(0).rolling(14).sum()
    sm_raw   = df['soil_moisture']
    # Combine actual SM reading with rolling rainfall evidence
    df['saturation_idx'] = (sm_raw * 0.6 + (rain_14d / (rain_14d.max() + 1)) * 0.4)
    df['saturation_idx'] = df['saturation_idx'].clip(0, 1).fillna(0)

    # ── PHYSICS FEATURE 5: Rainfall Anomaly ──
    # How much more/less rain than seasonal normal?
    if seasonal_rain_mean is not None and seasonal_rain_mean > 0:
        df['rain_anomaly'] = df['rainfall_mm'] / seasonal_rain_mean
    else:
        monthly_mean = df['rainfall_mm'].groupby(df.index.month).transform('mean')
        df['rain_anomaly'] = df['rainfall_mm'] / (monthly_mean + 1)
    df['rain_anomaly'] = df['rain_anomaly'].fillna(1.0).clip(0, 20)

    # ── PHYSICS FEATURE 6: Antecedent Precipitation Index ──
    weights   = np.array([0.5**i for i in range(30)])
    rain_vals = df['rainfall_mm'].fillna(0).values
    api_vals  = np.zeros(len(df))
    for i in range(1, len(df)):
        window  = rain_vals[max(0, i - 30):i][::-1]
        w       = weights[:len(window)]
        api_vals[i] = np.dot(window, w)
    df['antecedent_precip_idx'] = api_vals

    # ── Lag and rolling features (universal, no history needed) ──
    for lag in [1, 2, 3, 5, 7, 10, 14]:
        df[f'rain_lag{lag}d']  = df['rainfall_mm'].shift(lag)
        df[f'wl_lag{lag}d']    = df['water_level_m'].shift(lag)
    for w in [3, 5, 7, 14, 30]:
        df[f'rain_{w}d_sum']   = df['rainfall_mm'].rolling(w).sum()
        df[f'rain_{w}d_max']   = df['rainfall_mm'].rolling(w).max()
    df['wl_7d_mean']   = df['water_level_m'].rolling(7).mean()
    df['wl_7d_std']    = df['water_level_m'].rolling(7).std().fillna(0)
    df['wl_trend_7d']  = df['water_level_m'] - df['water_level_m'].shift(7)
    df['wl_change_1d'] = df['water_level_m'].diff(1)

    # ── Seasonal encoding (sine/cosine — works everywhere) ──
    df['month']       = df.index.month
    df['doy']         = df.index.dayofyear
    df['month_sin']   = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos']   = np.cos(2 * np.pi * df['month'] / 12)
    df['doy_sin']     = np.sin(2 * np.pi * df['doy'] / 365)
    df['doy_cos']     = np.cos(2 * np.pi * df['doy'] / 365)
    df['is_monsoon']  = df['month'].between(6, 9).astype(int)

    # ── Temperature features (heat drives soil evaporation) ──
    df['temp_range']  = df.get('temp_max', pd.Series(30, index=df.index)) \
                      - df.get('temp_min', pd.Series(20, index=df.index))

    df = df.dropna(subset=['rainfall_mm', 'water_level_m'])
    df = df.fillna(method='ffill').fillna(0)
    return df

In [ ]:
def compute_dynamic_risk_score(df_today, df_history):
    """
    Computes a 0-1 flood risk score from current conditions
    compared to seasonal norms. No static labels.

    Parameters:
        df_today   : single-row DataFrame for the date being assessed
        df_history : full historical DataFrame for the same station
                     (used to compute seasonal percentiles)
    Returns:
        float in [0, 1]
    """
    today_month = df_today.index[0].month

    # Seasonal subset for percentile calculation
    same_month_hist = df_history[df_history.index.month == today_month]
    if len(same_month_hist) < 10:
        same_month_hist = df_history   # fall back to full history if too few

    def safe_pct(series, pct):
        v = series.dropna()
        return float(np.percentile(v, pct)) if len(v) > 0 else 1.0

    wl_median  = safe_pct(same_month_hist['water_level_m'], 50)
    rain7_p90  = safe_pct(same_month_hist.get('rain_7d_sum',
                   same_month_hist['rainfall_mm'].rolling(7).sum()), 90)
    api_p90    = safe_pct(same_month_hist.get('antecedent_precip_idx',
                   pd.Series([1])), 90)

    # Component 1: Water level anomaly
    wl_now  = float(df_today['water_level_m'].iloc[0])
    c1      = np.clip(wl_now / (wl_median + 0.01) - 0.5, 0, 3) / 3.0

    # Component 2: Rainfall intensity
    rain7_now = float(df_today.get('rain_7d_sum', df_today['rainfall_mm']).iloc[0])
    c2        = np.clip(rain7_now / (rain7_p90 + 0.01), 0, 2) / 2.0

    # Component 3: Soil/catchment saturation
    api_now = float(df_today.get('antecedent_precip_idx', pd.Series([0])).iloc[0])
    c3      = np.clip(api_now / (api_p90 + 0.01), 0, 1)

    # Component 4: Upstream pressure (using available upstream lag features)
    upstream_cols = [c for c in df_today.columns if c.startswith('up_')]
    if upstream_cols:
        up_vals = df_today[upstream_cols].iloc[0].values
        c4      = np.clip(float(up_vals.mean()) / 50.0, 0, 1)
    else:
        c4 = c2 * 0.5   # proxy from local rain if no upstream data

    score = 0.40 * c1 + 0.30 * c2 + 0.20 * c3 + 0.10 * c4
    return float(np.clip(score, 0, 1))


def risk_level_from_score(score):
    """Converts 0-1 score to human-readable level and colour code."""
    if score < 0.20:   return "SAFE",    "\033[92m",  "green"
    if score < 0.45:   return "WATCH",   "\033[93m",  "yellow"
    if score < 0.68:   return "WARNING", "\033[33m",  "orange"
    if score < 0.85:   return "DANGER",  "\033[91m",  "red"
    return                    "EXTREME", "\033[41m",  "darkred"

In [ ]:
def fetch_weather(lat, lon, start_date, end_date, name=""):
    """Calls Open-Meteo Archive. Free, no key. Returns daily DataFrame."""
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start_date, 'end_date': end_date,
        'daily': ['precipitation_sum', 'temperature_2m_max',
                  'temperature_2m_min', 'wind_speed_10m_max',
                  'et0_fao_evapotranspiration'],
        'hourly': 'soil_moisture_0_to_7cm',
        'timezone': 'Asia/Dhaka', 'models': 'era5',
    }
    try:
        r = requests.get(METEO_ARCHIVE, params=params, timeout=45)
        r.raise_for_status()
        data = r.json()
        d = data['daily']
        df = pd.DataFrame({
            'date':             pd.to_datetime(d['time']),
            'rainfall_mm':      d['precipitation_sum'],
            'temp_max':         d['temperature_2m_max'],
            'temp_min':         d['temperature_2m_min'],
            'wind_speed':       d['wind_speed_10m_max'],
            'evapotranspiration': d['et0_fao_evapotranspiration'],
        })
        if 'hourly' in data:
            h = pd.DataFrame({'dt': pd.to_datetime(data['hourly']['time']),
                               'sm': data['hourly']['soil_moisture_0_to_7cm']})
            h['date'] = h['dt'].dt.normalize()
            df = df.merge(h.groupby('date')['sm'].mean().reset_index()
                          .rename(columns={'sm': 'soil_moisture'}), on='date', how='left')
        df['station'] = name
        return df.set_index('date')
    except Exception as e:
        print(f"  [WEATHER WARN] {name}: {e}")
        return None


def fetch_forecast(lat, lon, name="", days=7):
    """Calls Open-Meteo Forecast API. Free, no key."""
    params = {
        'latitude': lat, 'longitude': lon,
        'daily': ['precipitation_sum', 'temperature_2m_max',
                  'wind_speed_10m_max', 'precipitation_probability_max'],
        'forecast_days': days, 'timezone': 'Asia/Dhaka',
    }
    try:
        r = requests.get(METEO_FORECAST, params=params, timeout=30)
        r.raise_for_status()
        d = r.json()['daily']
        n = len(d['time'])
        df = pd.DataFrame({
            'date':             pd.to_datetime(d['time']),
            'rainfall_mm':      d['precipitation_sum'],
            'temp_max':         d['temperature_2m_max'],
            'wind_speed':       d['wind_speed_10m_max'],
            'rain_probability': d.get('precipitation_probability_max', [50]*n),
        })
        df['station'] = name
        return df.set_index('date')
    except Exception as e:
        print(f"  [FORECAST WARN] {name}: {e}")
        return None


def simulate_water_level(df, elevation_m):
    """
    Creates physics-consistent river levels from rainfall + elevation.
    Version 2: danger threshold derived purely from elevation physics,
    NOT from a static risk_zone label.

    PHYSICS:
    - Danger threshold = 3× the elevation above sea level
      (a 2m point floods at 6m river level; a 15m point at 45m)
    - Seasonal baseline follows real monsoon hydrology
    - Lagged rainfall feeds in with physically motivated delay weights
    """
    np.random.seed(42)
    n   = len(df)
    doy = df.index.dayofyear.values

    seasonal = 2.0 + 1.8 * np.sin(2 * np.pi * (doy - 80) / 365)

    rain = df['rainfall_mm'].fillna(0).values
    lagged = np.zeros(n)
    for lag, w in [(2, 0.35), (3, 0.40), (4, 0.15), (5, 0.10)]:
        pad = np.pad(rain, (lag, 0), mode='constant')[:n]
        lagged += w * pad
    max_r = lagged.max()
    if max_r > 0:
        lagged = (lagged / max_r) * (5.0 * (1.0 / (1 + elevation_m / 10.0)))

    water_level = seasonal + lagged + np.random.normal(0, 0.15, n)

    # Danger threshold: physics-derived, NOT a static zone label
    danger_thresh = max(elevation_m * 0.8 + 2.5, 3.5)   # lower elev → lower threshold

    df = df.copy()
    df['water_level_m']  = water_level
    df['danger_thresh']  = danger_thresh
    df['flood_occurred'] = (water_level > danger_thresh).astype(int)
    df['flood_severity'] = np.clip(
        np.digitize(water_level, [0, danger_thresh * 0.6, danger_thresh * 0.85,
                                  danger_thresh]) - 1, 0, 3
    )
    return df

In [ ]:
def load_all_data():
    print("\n" + "=" * 60)
    print(" STEP 1: OSM River Discovery")
    print("=" * 60)
    stations = fetch_osm_rivers()
    stations = fetch_elevations(stations)
    stations = compute_slope(stations)
    print(f"  Total stations discovered + enriched: {len(stations)}")

    print("\n" + "=" * 60)
    print(" STEP 2: Upstream Catchment Data (India / Nepal)")
    print("=" * 60)
    upstream_data = {}
    for name, info in UPSTREAM_FIXED.items():
        print(f"  Fetching upstream {name} ...")
        df = fetch_weather(info['lat'], info['lon'], TRAIN_START, VAL_END, name)
        if df is not None:
            upstream_data[name] = df
        time.sleep(0.4)

    print("\n" + "=" * 60)
    print(" STEP 3: Bangladesh Station Weather Data")
    print("=" * 60)
    station_data = {}
    for st in stations[:20]:   # limit to 20 for notebook speed; remove cap for full run
        label = f"{st['name']}_{st['lat']:.2f}"
        print(f"  Fetching {label} (elev={st['elevation_m']:.0f}m, slope={st['slope']:.5f}) ...")
        df = fetch_weather(st['lat'], st['lon'], TRAIN_START, VAL_END, label)
        if df is not None:
            df = simulate_water_level(df, st['elevation_m'])
            st_info = {**st, 'label': label}
            station_data[label] = (df, st_info)
        time.sleep(0.4)

    print(f"\nLoaded: {len(station_data)} stations, {len(upstream_data)} upstream catchments")
    return station_data, upstream_data, stations


station_data, upstream_data, all_stations = load_all_data()


 STEP 1: OSM River Discovery
  Querying OSM Overpass API for Bangladesh rivers ...
  Overpass failed: 406 Client Error: Not Acceptable for url: https://overpass-api.de/api/interpreter. Using fallback stations.
  Fetching SRTM elevations from OpenTopoData ...
  Total stations discovered + enriched: 12

 STEP 2: Upstream Catchment Data (India / Nepal)
  Fetching upstream Assam_Dhubri ...
  Fetching upstream Meghalaya_Shillong ...
  Fetching upstream Nepal_Birganj ...

 STEP 3: Bangladesh Station Weather Data
  Fetching Brahmaputra_25.81 (elev=34m, slope=0.00012) ...
  Fetching Brahmaputra_25.33 (elev=25m, slope=0.00007) ...
  Fetching Jamuna_24.94 (elev=25m, slope=0.00008) ...
  [WEATHER WARN] Jamuna_24.94: 429 Client Error: Too Many Requests for url: https://archive-api.open-meteo.com/v1/archive?latitude=24.937&longitude=89.937&start_date=2015-01-01&end_date=2023-12-31&daily=precipitation_sum&daily=temperature_2m_max&daily=temperature_2m_min&daily=wind_speed_10m_max&daily=et0_fao_evapo